In [ ]:
!python -m pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
!python -m pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
!python -m pip install rouge

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
import os

os.chdir("/content/drive/MyDrive/distractor_analysis")

with open("data_dict.json", 'r', encoding="utf8") as inp:
  data_dict = json.load(inp)

true_distractors = data_dict["true_distractors"]

In [ ]:
os.chdir("/content/drive/MyDrive/RuRACE/OrganizedOutputData1")

In [ ]:
import re

import pandas as pd

from typing import List, Any, Union
from rouge import Rouge

from evaluate import load

In [ ]:
# metrics:
bleu = load("bleu")
meteor = load("meteor")
bertscore = load("bertscore")
rouge = Rouge()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
def compute_metrics(output: List[str], label_batch: List[str]) -> dict:
    rouge_scores = []
    for sent_a, sent_b in zip(output, label_batch):
      try:
        scores_ = rouge.get_scores(sent_a, sent_b)
        scores = {
            "rouge1": scores_[0]["rouge-1"]["f"],
            "rouge2": scores_[0]["rouge-2"]["f"],
            "rougeL": scores_[0]["rouge-l"]["f"]
        }
      except Exception as exc:
        print(exc)
        scores = {
            "rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0
        }
      rouge_scores.append(scores)

    rouge_scores = pd.DataFrame(rouge_scores).mean()

    metric_dict = {
        "bleu1": bleu.compute(predictions=output, references=[[label] for label in label_batch], max_order=1)["bleu"],
        "bleu2": bleu.compute(predictions=output, references=[[label] for label in label_batch], max_order=2)["bleu"],
        "bleu3": bleu.compute(predictions=output, references=[[label] for label in label_batch], max_order=3)["bleu"],
        "bleu4": bleu.compute(predictions=output, references=[[label] for label in label_batch], max_order=4)["bleu"],
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "meteor": meteor.compute(predictions=output, references=label_batch)["meteor"],
        "bertscore": pd.DataFrame(bertscore.compute(
                        predictions=output,
                        references=label_batch,
                        lang="ru", verbose=True,
                        model_type = "bert-base-multilingual-cased"
                    ))["f1"].mean()
    }
    return metric_dict


def remove_duplicates(l: list) -> list:
    outp = []

    for item in l:
        if item not in outp:
            outp.append(item)

    return outp


def preprocess_values(s: str) -> str:
  if type(s) != str:
    s = ""
  s = s.strip()
  regex1 = '("(.*)";) *+"(.*)"'
  regex2 = '((.*)\n)+(.*)'

  if re.match(regex1, s):
    options = [option.strip('" ').strip() for option in s.split(';')]
    options = [option for option in options if option]
    options = remove_duplicates(options)
    return '\n'.join(options)
  elif re.match(regex2, s):
    options = [option.strip('" ') for option in s.split('\n')]
    options = [option for option in options if option]
    options = remove_duplicates(options)
    return '\n'.join(options)
  else:
    return s.strip('" ')


def recalculate_metrics(input_addr: Union[str, pd.DataFrame], ege:bool=True) -> Any:
    if type(input_addr) == str:
      if input_addr.endswith(".xlsx"):
        df = pd.read_excel(input_addr, engine="openpyxl")
      elif input_addr.endswith(".csv"):
        df = pd.read_csv(input_addr, sep=";")
    else:
      df = input_addr

    output = df["output"].apply(preprocess_values).values

    if ege:
      label_batch = ['\n'.join(item) for item in true_distractors]
    else:
      label_batch = df["distractors"].apply(preprocess_values).values

    with open(f"{input_addr}_outp", 'w', encoding="utf8") as outp:
      json.dump(output.tolist(), outp, ensure_ascii=False, indent=2)

    if not ege:
      with open(f"{input_addr}_labels", 'w', encoding="utf8") as outp:
        json.dump(label_batch.tolist(), outp, ensure_ascii=False, indent=2)

    print(len(output), len(label_batch))

    scores = compute_metrics(
       output=output,
       label_batch=label_batch
    )

    return scores

In [ ]:
METRICS = dict()

In [ ]:
metrics_chatgpt = recalculate_metrics("Output_ChatGPT4o.csv")
METRICS["EGE/ChatGPT4o"] = metrics_chatgpt

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 818.18 seconds, 0.07 sentences/sec


In [ ]:
metrics_chatgpt

{'bleu1': 0.18471219721488005,
 'bleu2': 0.07052140176974657,
 'bleu3': 0.035667940423042895,
 'bleu4': 0.018700702821765544,
 'rouge1': np.float64(0.11166948851711038),
 'rouge2': np.float64(0.013118622106089852),
 'rougeL': np.float64(0.10587662496538132),
 'meteor': np.float64(0.115112085969925),
 'bertscore': np.float64(0.687595930966464)}

In [ ]:
metrics_deepseek = recalculate_metrics("Output_DeepSeek.csv")
METRICS["EGE/DeepSeek"] = metrics_deepseek

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 852.31 seconds, 0.06 sentences/sec


In [ ]:
metrics_deepseek

{'bleu1': 0.22539812168231935,
 'bleu2': 0.08176025694354173,
 'bleu3': 0.034998425819921145,
 'bleu4': 0.011704755824973517,
 'rouge1': np.float64(0.122951463200913),
 'rouge2': np.float64(0.014587536883805945),
 'rougeL': np.float64(0.11773751122166906),
 'meteor': np.float64(0.14322093717997814),
 'bertscore': np.float64(0.6898899273438888)}

In [ ]:
import os
os.chdir("/content/drive/MyDrive/RuRACE/OrganizedOutputData")

In [ ]:
os.listdir()

['MuSeRC', 'Ru-RACE-TF', 'Ru-RACE-TITLE', 'BartDG-EGE', 'EGE']

In [ ]:
METRICS["EGE/RuT5-RACE-TF"] = recalculate_metrics("EGE/MetricsEGET5.xlsx")
METRICS["EGE/RuT5-RACE-TF"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 885.53 seconds, 0.06 sentences/sec


{'bleu1': 0.1587256222294004,
 'bleu2': 0.05104621718236795,
 'bleu3': 0.019790778045937592,
 'bleu4': 0.007644703337625666,
 'rouge1': np.float64(0.08279255755804371),
 'rouge2': np.float64(0.005329763306813829),
 'rougeL': np.float64(0.07890715806434145),
 'meteor': np.float64(0.09942480884142213),
 'bertscore': np.float64(0.6718651045452465)}

In [ ]:
METRICS["EGE/RuT5-MuSeRC"] = recalculate_metrics("EGE/metrics_muserc_ege_t5.csv")
METRICS["EGE/RuT5-MuSeRC"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 916.19 seconds, 0.06 sentences/sec


{'bleu1': 0.037748921372173894,
 'bleu2': 0.011685821546660487,
 'bleu3': 0.0030397649293690235,
 'bleu4': 0.0,
 'rouge1': np.float64(0.06542077821076839),
 'rouge2': np.float64(0.005614732244648949),
 'rougeL': np.float64(0.06325535742465796),
 'meteor': np.float64(0.05885253404971037),
 'bertscore': np.float64(0.6490073399110274)}

In [ ]:
METRICS["EGE/RuGPT3-RACE-TF"] = recalculate_metrics("EGE/MetricsEGERuGPT3.xlsx")
METRICS["EGE/RuGPT3-RACE-TF"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 948.24 seconds, 0.06 sentences/sec


{'bleu1': 0.1501091082928315,
 'bleu2': 0.0363779703472513,
 'bleu3': 0.008310246560044782,
 'bleu4': 0.0,
 'rouge1': np.float64(0.07043838473244358),
 'rouge2': np.float64(0.003093495751243604),
 'rougeL': np.float64(0.0680047732684768),
 'meteor': np.float64(0.0921320328470214),
 'bertscore': np.float64(0.6573472239754417)}

In [ ]:
METRICS["EGE/RuGPT3-MuSeRC"] = recalculate_metrics("EGE/metrics_muserc_ege_gpt3.csv")
METRICS["EGE/RuGPT3-MuSeRC"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 978.08 seconds, 0.06 sentences/sec


{'bleu1': 0.07361688292359823,
 'bleu2': 0.014608850499381482,
 'bleu3': 0.004030983818541121,
 'bleu4': 0.0,
 'rouge1': np.float64(0.04973432555551742),
 'rouge2': np.float64(0.0042509668894110415),
 'rougeL': np.float64(0.049191584714269106),
 'meteor': np.float64(0.06819763746622477),
 'bertscore': np.float64(0.6285143917257136)}

In [ ]:
METRICS["EGE/Базовая RuGPT3"] = recalculate_metrics("EGE/metrics_baseline_ege_gpt3.csv")
METRICS["EGE/Базовая RuGPT3"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1022.40 seconds, 0.05 sentences/sec


{'bleu1': 0.11255259467040675,
 'bleu2': 0.02373534453936712,
 'bleu3': 0.005900758989608143,
 'bleu4': 0.0,
 'rouge1': np.float64(0.049963064002776066),
 'rouge2': np.float64(0.001471562440319376),
 'rougeL': np.float64(0.04740375442464361),
 'meteor': np.float64(0.07837376315276799),
 'bertscore': np.float64(0.5571652228182012)}

In [ ]:
METRICS["EGE/BART-DG-ANPM"] = recalculate_metrics("BartDG-EGE/BartDG_ANPM_OutputEGE.xlsx")
METRICS["EGE/BART-DG-ANPM"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1054.18 seconds, 0.05 sentences/sec


{'bleu1': 0.14222265084924404,
 'bleu2': 0.03353438820798083,
 'bleu3': 0.007850996033497199,
 'bleu4': 0.0,
 'rouge1': np.float64(0.06346649828150405),
 'rouge2': np.float64(0.0010940572860067296),
 'rougeL': np.float64(0.06237493751252393),
 'meteor': np.float64(0.08836932993779605),
 'bertscore': np.float64(0.6640863201834939)}

In [ ]:
METRICS["EGE/BART-DG-PM"] = recalculate_metrics("BartDG-EGE/BartDG_PM_OutputEGE.xlsx")
METRICS["EGE/BART-DG-PM"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1086.88 seconds, 0.05 sentences/sec


{'bleu1': 0.15500083332621617,
 'bleu2': 0.03611885784217797,
 'bleu3': 0.00827664003807247,
 'bleu4': 0.0,
 'rouge1': np.float64(0.07537358506272081),
 'rouge2': np.float64(0.004010681768734127),
 'rougeL': np.float64(0.0718241155164317),
 'meteor': np.float64(0.09478141813654166),
 'bertscore': np.float64(0.6675226883454757)}

In [ ]:
METRICS["EGE/BART-DG"] = recalculate_metrics("BartDG-EGE/BartDGOutputEGE.xlsx")
METRICS["EGE/BART-DG"]

55 55
calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1118.52 seconds, 0.05 sentences/sec


{'bleu1': 0.13603006499736844,
 'bleu2': 0.036954289029787604,
 'bleu3': 0.010440065979782307,
 'bleu4': 0.0,
 'rouge1': np.float64(0.0721468949854847),
 'rouge2': np.float64(0.0035331989275387337),
 'rougeL': np.float64(0.07120222561976271),
 'meteor': np.float64(0.08983608666276031),
 'bertscore': np.float64(0.6661410169167952)}

In [ ]:
os.listdir("Ru-RACE-TF")

['T5Metrics-TF.xlsx',
 'T5Metrics-TF-Baseline.xlsx',
 'RuGPT3Metrics-TF.xlsx',
 'RuGPT3Metrics-TF-Baseline.xlsx',
 'T5Metrics-TF-Baseline-val.xlsx',
 'T5Metrics-TF-val.xlsx',
 'RuGPT3Metrics-TF-Baseline-val.xlsx',
 'RuGPT3Metrics-TFwithBertScore.xlsx',
 'RuGPT3Metrics-TF-valwithBertScore.xlsx',
 'T5Metrics-TFwithBertScore.xlsx',
 'T5Metrics-TF-valwithBertScore.xlsx',
 'RuGPT3Metrics-TF-BaselinewithBertScore.xlsx',
 'RuGPT3Metrics-TF-Baseline-valwithBertScore.xlsx',
 'T5Metrics-TF-BaselinewithBertScore.xlsx',
 'T5Metrics-TF-Baseline-valwithBertScore.xlsx',
 'RuGPT3Metrics-TF-val.xlsx',
 'T5Metrics-TF.xlsx_outp',
 'T5Metrics-TF.xlsx_labels',
 'T5Metrics-TF-val.xlsx_outp',
 'T5Metrics-TF-val.xlsx_labels']

In [ ]:
METRICS["Ru-RACE-TF/RuT5-RACE-TF/тест"] = recalculate_metrics("Ru-RACE-TF/T5Metrics-TF.xlsx", ege=False)
METRICS["Ru-RACE-TF/RuT5-RACE-TF/тест"]

187 187
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1062.65 seconds, 0.18 sentences/sec


{'bleu1': 0.2815711252143785,
 'bleu2': 0.1614740139942107,
 'bleu3': 0.11439148114991546,
 'bleu4': 0.0889693104592183,
 'rouge1': np.float64(0.19141163784178003),
 'rouge2': np.float64(0.07745715025072787),
 'rougeL': np.float64(0.1866069423251465),
 'meteor': np.float64(0.20659306571017452),
 'bertscore': np.float64(0.7254415332952285)}

In [ ]:
METRICS["Ru-RACE-TF/RuT5-RACE-TF/разработка"] = recalculate_metrics("Ru-RACE-TF/T5Metrics-TF-val.xlsx", ege=False)
METRICS["Ru-RACE-TF/RuT5-RACE-TF/разработка"]

175 175
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1141.46 seconds, 0.15 sentences/sec


{'bleu1': 0.3120278308402592,
 'bleu2': 0.2036033386959361,
 'bleu3': 0.15958493135322924,
 'bleu4': 0.13586642233699464,
 'rouge1': np.float64(0.22160315545081988),
 'rouge2': np.float64(0.11702506206801551),
 'rougeL': np.float64(0.21900402058017504),
 'meteor': np.float64(0.23912224273280994),
 'bertscore': np.float64(0.7350727902139936)}

In [ ]:
METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/тест"] = recalculate_metrics("Ru-RACE-TF/RuGPT3Metrics-TF.xlsx", ege=False)
METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/тест"]

187 187
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1203.07 seconds, 0.16 sentences/sec


{'bleu1': 0.23895967882702043,
 'bleu2': 0.11201483583065228,
 'bleu3': 0.0697782178843019,
 'bleu4': 0.048903404235321246,
 'rouge1': np.float64(0.14183032148772903),
 'rouge2': np.float64(0.04298818755327981),
 'rougeL': np.float64(0.13795784771775013),
 'meteor': np.float64(0.16819202192706406),
 'bertscore': np.float64(0.7021650005789364)}

In [ ]:
METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/разработка"] = recalculate_metrics("Ru-RACE-TF/RuGPT3Metrics-TF-val.xlsx", ege=False)
METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/разработка"]

175 175
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1282.26 seconds, 0.14 sentences/sec


{'bleu1': 0.26225165562913905,
 'bleu2': 0.1466410902043583,
 'bleu3': 0.10659978615152267,
 'bleu4': 0.08756257142539359,
 'rouge1': np.float64(0.1662240227911487),
 'rouge2': np.float64(0.07044411153732268),
 'rougeL': np.float64(0.161641919161681),
 'meteor': np.float64(0.18916624845658309),
 'bertscore': np.float64(0.7101228414263044)}

In [ ]:
METRICS["Ru-RACE-TF/Базовая RuGPT3/тест"] = recalculate_metrics("Ru-RACE-TF/RuGPT3Metrics-TF-Baseline.xlsx", ege=False)
METRICS["Ru-RACE-TF/Базовая RuGPT3/тест"]

187 187
Hypothesis is empty.
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1358.10 seconds, 0.14 sentences/sec


{'bleu1': 0.14930733709594665,
 'bleu2': 0.055787256101667744,
 'bleu3': 0.028330133651257677,
 'bleu4': 0.01733396436679089,
 'rouge1': np.float64(0.08531368667895854),
 'rouge2': np.float64(0.017611553186290835),
 'rougeL': np.float64(0.0829273760442251),
 'meteor': np.float64(0.09417182090232078),
 'bertscore': np.float64(0.6367405783683858)}

In [ ]:
METRICS["Ru-RACE-TF/Базовая RuGPT3/разработка"] = recalculate_metrics("Ru-RACE-TF/RuGPT3Metrics-TF-Baseline-val.xlsx", ege=False)
METRICS["Ru-RACE-TF/Базовая RuGPT3/разработка"]

175 175
Hypothesis is empty.
calculating scores...
computing bert embedding.


  0%|          | 0/6 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/3 [00:00<?, ?it/s]

done in 1447.75 seconds, 0.12 sentences/sec


{'bleu1': 0.15004766444232603,
 'bleu2': 0.05412344250099221,
 'bleu3': 0.024961538870444482,
 'bleu4': 0.01232178255855408,
 'rouge1': np.float64(0.0854662594254064),
 'rouge2': np.float64(0.016669269419184313),
 'rougeL': np.float64(0.08302963458819616),
 'meteor': np.float64(0.09538927765709529),
 'bertscore': np.float64(0.6331411589894976)}

In [ ]:
os.listdir("Ru-RACE-TITLE")

['T5Metrics-Title-Baseline.xlsx',
 'T5Metrics-Title.xlsx',
 'RuGPT3Metrics-Title-Baseline.xlsx',
 'RuGPT3Metrics-Title.xlsx',
 'RuGPT3Metrics-Title-Baseline-val.xlsx',
 'RuGPT3Metrics-Title-val.xlsx',
 'T5Metrics-Title-Baseline-val.xlsx',
 'T5Metrics-Title-val.xlsx',
 'RuGPT3Metrics-TitlewithBertScore.xlsx',
 'RuGPT3Metrics-Title-valwithBertScore.xlsx',
 'T5Metrics-TitlewithBertScore.xlsx',
 'T5Metrics-Title-valwithBertScore.xlsx',
 'RuGPT3Metrics-Title-BaselinewithBertScore.xlsx',
 'RuGPT3Metrics-Title-Baseline-valwithBertScore.xlsx',
 'T5Metrics-Title-BaselinewithBertScore.xlsx',
 'T5Metrics-Title-Baseline-valwithBertScore.xlsx',
 'T5Metrics-Title.xlsx_outp',
 'T5Metrics-Title.xlsx_labels',
 'T5Metrics-Title-val.xlsx_outp',
 'T5Metrics-Title-val.xlsx_labels']

In [ ]:
METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/тест"] = recalculate_metrics("Ru-RACE-TITLE/T5Metrics-Title.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/тест"]

242 242
calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1435.21 seconds, 0.17 sentences/sec


{'bleu1': 0.19284789110351494,
 'bleu2': 0.12630089501009745,
 'bleu3': 0.09483205710993466,
 'bleu4': 0.0763950304947508,
 'rouge1': np.float64(0.17258958383516124),
 'rouge2': np.float64(0.08297198052715832),
 'rougeL': np.float64(0.16044925966397566),
 'meteor': np.float64(0.15830456406488969),
 'bertscore': np.float64(0.7157990996502648)}

In [ ]:
METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/разработка"] = recalculate_metrics("Ru-RACE-TITLE/T5Metrics-Title-val.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/разработка"]

219 219
calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1503.82 seconds, 0.15 sentences/sec


{'bleu1': 0.19997404460398524,
 'bleu2': 0.14078400177151912,
 'bleu3': 0.11316580478074445,
 'bleu4': 0.09801487952163826,
 'rouge1': np.float64(0.1762985485933401),
 'rouge2': np.float64(0.09807078902805261),
 'rougeL': np.float64(0.17074346237180088),
 'meteor': np.float64(0.1648383210725985),
 'bertscore': np.float64(0.718433371689766)}

In [ ]:
METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/тест"] = recalculate_metrics("Ru-RACE-TITLE/RuGPT3Metrics-Title.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/тест"]

242 242
calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1643.17 seconds, 0.15 sentences/sec


{'bleu1': 0.14577938891765924,
 'bleu2': 0.08225236959116121,
 'bleu3': 0.050681584329712426,
 'bleu4': 0.03191493832461374,
 'rouge1': np.float64(0.12793514645224374),
 'rouge2': np.float64(0.04777516368199007),
 'rougeL': np.float64(0.12604396640903068),
 'meteor': np.float64(0.12396325677146723),
 'bertscore': np.float64(0.6868250911886041)}

In [ ]:
METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/разработка"] = recalculate_metrics("Ru-RACE-TITLE/RuGPT3Metrics-Title-val.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/разработка"]

219 219
calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1785.69 seconds, 0.12 sentences/sec


{'bleu1': 0.1429794520547945,
 'bleu2': 0.07778153818275273,
 'bleu3': 0.05193734571035541,
 'bleu4': 0.03835978069034958,
 'rouge1': np.float64(0.12717800878189098),
 'rouge2': np.float64(0.048690815182771525),
 'rougeL': np.float64(0.12316338367435341),
 'meteor': np.float64(0.1278027671870394),
 'bertscore': np.float64(0.6873025654657791)}

In [ ]:
METRICS["Ru-RACE-TITLE/Базовая RuGPT3/тест"] = recalculate_metrics("Ru-RACE-TITLE/RuGPT3Metrics-Title-Baseline.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/Базовая RuGPT3/тест"]

242 242
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1846.65 seconds, 0.13 sentences/sec


{'bleu1': 0.07643312101910828,
 'bleu2': 0.032399726364070326,
 'bleu3': 0.01423530826981923,
 'bleu4': 0.005385982698161673,
 'rouge1': np.float64(0.046247780631038064),
 'rouge2': np.float64(0.012804004490250558),
 'rougeL': np.float64(0.04550898774348485),
 'meteor': np.float64(0.055131609916056666),
 'bertscore': np.float64(0.5772593270156009)}

In [ ]:
METRICS["Ru-RACE-TITLE/Базовая RuGPT3/разработка"] = recalculate_metrics("Ru-RACE-TITLE/RuGPT3Metrics-Title-Baseline-val.xlsx", ege=False)
METRICS["Ru-RACE-TITLE/Базовая RuGPT3/разработка"]

219 219
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 1918.61 seconds, 0.11 sentences/sec


{'bleu1': 0.07281964436917868,
 'bleu2': 0.026823040994608947,
 'bleu3': 0.011120525760161397,
 'bleu4': 0.004646634218962105,
 'rouge1': np.float64(0.045824520142843765),
 'rouge2': np.float64(0.009067824306886345),
 'rougeL': np.float64(0.043132676043347334),
 'meteor': np.float64(0.05363911738271871),
 'bertscore': np.float64(0.583729686258046)}

In [ ]:
os.listdir("MuSeRC")

['metrics_muserc_val_gpt3.csv',
 'metrics_muserc_val_t5_baseline.csv',
 'metrics_muserc_val_gpt3_baseline.csv',
 'metrics_muserc_val_t5.csv',
 'metrics_muserc_train_gpt3_baseline.csv',
 'metrics_muserc_train_t5_baseline.csv',
 'metrics_muserc_train_gpt3.csv',
 'metrics_muserc_train_t5.csv',
 'metrics_muserc_val_t5withBertScore.xlsx',
 'metrics_muserc_val_gpt3withBertScore.xlsx',
 'metrics_muserc_val_gpt3_baselinewithBertScore.xlsx',
 'metrics_muserc_val_t5_baselinewithBertScore.xlsx',
 'metrics_muserc_val_t5.csv_labels',
 'metrics_muserc_val_t5.csv_outp',
 'metrics_muserc_val_gpt3.csv_outp',
 'metrics_muserc_val_gpt3.csv_labels',
 'metrics_muserc_val_gpt3_baseline.csv_outp',
 'metrics_muserc_val_gpt3_baseline.csv_labels']

In [ ]:
METRICS["MuSeRC/RuT5-MuSeRC-DG"] = recalculate_metrics("MuSeRC/metrics_muserc_val_t5.csv", ege=False)
METRICS["MuSeRC/RuT5-MuSeRC-DG"]

528 528
calculating scores...
computing bert embedding.


  0%|          | 0/17 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 1712.33 seconds, 0.31 sentences/sec


{'bleu1': 0.2455595602126905,
 'bleu2': 0.1406926326126442,
 'bleu3': 0.09386846482616942,
 'bleu4': 0.06687537045903555,
 'rouge1': np.float64(0.14781081758882336),
 'rouge2': np.float64(0.050818800956455454),
 'rougeL': np.float64(0.14717729706893923),
 'meteor': np.float64(0.20525653617314124),
 'bertscore': np.float64(0.715026838874275)}

In [ ]:
METRICS["MuSeRC/RuGPT3-MuSeRC-DG"] = recalculate_metrics("MuSeRC/metrics_muserc_val_gpt3.csv", ege=False)
METRICS["MuSeRC/RuGPT3-MuSeRC-DG"]

529 529
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
Hypothesis is empty.
calculating scores...
computing bert embedding.


  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 1808.22 seconds, 0.29 sentences/sec


{'bleu1': 0.22751942488217514,
 'bleu2': 0.09809209252206129,
 'bleu3': 0.05288999668117933,
 'bleu4': 0.031052073345391087,
 'rouge1': np.float64(0.09219786690294944),
 'rouge2': np.float64(0.017036732423481624),
 'rougeL': np.float64(0.091894059179482),
 'meteor': np.float64(0.16939755774543183),
 'bertscore': np.float64(0.6662289929525163)}

In [ ]:
METRICS["MuSeRC/Базовая RuGPT3"] = recalculate_metrics("MuSeRC/metrics_muserc_val_gpt3_baseline.csv", ege=False)
METRICS["MuSeRC/Базовая RuGPT3"]

528 528
calculating scores...
computing bert embedding.


  0%|          | 0/17 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/9 [00:00<?, ?it/s]

done in 1978.02 seconds, 0.27 sentences/sec


{'bleu1': 0.17678255745433116,
 'bleu2': 0.09433746320103943,
 'bleu3': 0.06593941416898143,
 'bleu4': 0.05113111829233242,
 'rouge1': np.float64(0.10315328917410772),
 'rouge2': np.float64(0.03221839359389249),
 'rougeL': np.float64(0.10100524786328487),
 'meteor': np.float64(0.14695231268805908),
 'bertscore': np.float64(0.6360201968839674)}

In [ ]:
def interleave_columns(
    df1: pd.DataFrame, df2: pd.DataFrame,
    suffixes=("_1", "_2")
  ) -> pd.DataFrame:
    """
    Чередует столбцы двух DataFrame с одинаковыми названиями колонок.

    Параметры:
        df1, df2 : pd.DataFrame
            Датафреймы с одинаковыми именами колонок и одинаковой длиной (по строкам).
        suffixes : tuple(str, str), по умолчанию ("_1", "_2")
            Суффиксы для различения одноимённых столбцов.

    Возвращает:
        pd.DataFrame с чередующимися столбцами.
    """
    # Сбрасываем индексы, чтобы не было разнобоя
    df1 = df1.reset_index(drop=True).add_suffix(suffixes[0])
    df2 = df2.reset_index(drop=True).add_suffix(suffixes[1])

    # Формируем порядок колонок: A_1, A_2, B_1, B_2 ...
    cols = [c for pair in zip(df1.columns, df2.columns) for c in pair]

    return pd.concat([df1, df2], axis=1)[cols]

In [ ]:
columns_table4 = ["bleu4", "meteor", "rougeL", "bertscore"]

table4_part1_dev = pd.DataFrame([
    pd.Series(METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/разработка"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/разработка"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TITLE/Базовая RuGPT3/разработка"])[columns_table4]
])
table4_part1_dev["model"] = [
    "RuGPT3-RACE-TITLE",
    "RuT5-RACE-TITLE",
    "Базовая RuGPT3"
]
table4_part1_dev = table4_part1_dev.set_index("model")

table4_part1_test = pd.DataFrame([
    pd.Series(METRICS["Ru-RACE-TITLE/RuGPT3-RACE-TITLE/тест"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TITLE/RuT5-RACE-TITLE/тест"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TITLE/Базовая RuGPT3/тест"])[columns_table4]
])
table4_part1_test["model"] = [
    "RuGPT3-RACE-TITLE",
    "RuT5-RACE-TITLE",
    "Базовая RuGPT3"
]
table4_part1_test = table4_part1_test.set_index("model")

table4_part1 = interleave_columns(
    table4_part1_dev, table4_part1_test,
    suffixes = ["_paзработка", "_тест"]
)

In [ ]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
(table4_part1 * 100)

,bleu4_paзработка,bleu4_тест,meteor_paзработка,meteor_тест,rougeL_paзработка,rougeL_тест,bertscore_paзработка,bertscore_тест
0,3.84,3.19,12.78,12.40,12.32,12.60,68.73,68.68
1,9.80,7.64,16.48,15.83,17.07,16.04,71.84,71.58
2,0.46,0.54,5.36,5.51,4.31,4.55,58.37,57.73


In [ ]:
columns_table4 = ["bleu4", "meteor", "rougeL", "bertscore"]

table4_part2_dev = pd.DataFrame([
    pd.Series(METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/разработка"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TF/RuT5-RACE-TF/разработка"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TF/Базовая RuGPT3/разработка"])[columns_table4]
])
table4_part2_dev["model"] = [
    "RuGPT3-RACE-TF",
    "RuT5-RACE-TF",
    "Базовая RuGPT3"
]
table4_part2_dev = table4_part2_dev.set_index("model")

table4_part2_test = pd.DataFrame([
    pd.Series(METRICS["Ru-RACE-TF/RuGPT3-RACE-TF/тест"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TF/RuT5-RACE-TF/тест"])[columns_table4],
    pd.Series(METRICS["Ru-RACE-TF/Базовая RuGPT3/тест"])[columns_table4]
])
table4_part2_test["model"] = [
    "RuGPT3-RACE-TF",
    "RuT5-RACE-TF",
    "Базовая RuGPT3"
]
table4_part2_test = table4_part2_test.set_index("model")

table4_part2 = interleave_columns(
    table4_part2_dev, table4_part2_test,
    suffixes = ["_paзработка", "_тест"]
)

In [ ]:
(table4_part2 * 100)

,bleu4_paзработка,bleu4_тест,meteor_paзработка,meteor_тест,rougeL_paзработка,rougeL_тест,bertscore_paзработка,bertscore_тест
0,8.76,4.89,18.92,16.82,16.16,13.80,71.01,70.22
1,13.59,8.90,23.91,20.66,21.90,18.66,73.51,72.54
2,1.23,1.73,9.54,9.42,8.30,8.29,63.31,63.67


In [ ]:
columns_table5 = ["bleu4", "meteor", "rougeL", "bertscore"]

table5 = pd.DataFrame([
    pd.Series(METRICS["MuSeRC/RuGPT3-MuSeRC-DG"])[columns_table5],
    pd.Series(METRICS["MuSeRC/RuT5-MuSeRC-DG"])[columns_table5],
    pd.Series(METRICS["MuSeRC/Базовая RuGPT3"])[columns_table5]
])

In [ ]:
(table5 * 100)

,bleu4,meteor,rougeL,bertscore
0,3.11,16.94,9.19,66.62
1,6.69,20.53,14.72,71.50
2,5.11,14.70,10.10,63.60


In [ ]:
columns_table6 = [
    "bleu1", "bleu2", "bleu3", "bleu4",
    "meteor", "rougeL", "bertscore"
]

table6 = pd.DataFrame([
    pd.Series(METRICS["EGE/RuGPT3-RACE-TF"])[columns_table6],
    pd.Series(METRICS["EGE/RuT5-RACE-TF"])[columns_table6],
    pd.Series(METRICS["EGE/RuGPT3-MuSeRC"])[columns_table6],
    pd.Series(METRICS["EGE/RuT5-MuSeRC"])[columns_table6],
    pd.Series(METRICS["EGE/Базовая RuGPT3"])[columns_table6],
    pd.Series(METRICS["EGE/BART-DG"])[columns_table6],
    pd.Series(METRICS["EGE/BART-DG-PM"])[columns_table6],
    pd.Series(METRICS["EGE/BART-DG-ANPM"])[columns_table6]
])

In [ ]:
(table6 * 100)

,bleu1,bleu2,bleu3,bleu4,meteor,rougeL,bertscore
0,15.01,3.64,0.83,0.00,9.21,6.80,65.73
1,15.87,5.10,1.98,0.76,9.94,7.89,67.19
2,7.36,1.46,0.40,0.00,6.82,4.92,62.85
3,3.77,1.17,0.30,0.00,5.89,6.33,64.90
4,11.26,2.37,0.59,0.00,7.84,4.74,55.72
5,13.60,3.70,1.04,0.00,8.98,7.12,66.61
6,15.50,3.61,0.83,0.00,9.48,7.18,66.75
7,14.22,3.35,0.79,0.00,8.84,6.24,66.41
